# Week 7 Day 5 - Model Evaluation

## Objective
Compare Logistic Regression, Decision Tree and Random Forest using
5-fold TimeSeriesSplit.

Models are also compared against a naive baseline that always predicts
the market will go UP.

Evaluation Metrics

- Accuracy
- Precision
- Recall
- F1 Score

Finally, rank all models based on their average F1 Score.

In [62]:
# Numerical computing
import numpy as np

# Data handling
import pandas as pd

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Time-series split
from sklearn.model_selection import TimeSeriesSplit

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [63]:
import pandas as pd

data = pd.read_csv("../Day01/ml_dataset.csv")

print(data.columns.tolist())

['Price', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA_10', 'SMA_20', 'RSI', 'MACD', 'Signal', 'Return', 'Lag_1', 'Lag_5', 'Lag_21', 'Target']


In [ ]:
# Feature columns

feature_cols = [
    "SMA_20",
    "RSI",
    "MACD",
    "Lag_1",
    "Lag_5",
    "Lag_21"
]

# Feature matrix
X = data[feature_cols]

In [ ]:
# Target column

y = data["Target"]

In [ ]:
# Ensure target is binary (0 and 1)

y = y.astype(int)

print(y.unique())

In [ ]:
# Combine X and y

dataset = pd.concat([X, y], axis=1)

# Remove missing values

dataset = dataset.dropna()

# Split again

X = dataset[feature_cols]
y = dataset["Target"].astype(int)

## Time Series Cross Validation

In [ ]:
# Create TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

## Create Models

In [ ]:
models = {

    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Decision Tree":
        DecisionTreeClassifier(random_state=42),

    "Random Forest":
        RandomForestClassifier(
            random_state=42,
            n_estimators=100
        )
}

## Evaluate Models

In [ ]:
results = []

for model_name, model in models.items():

    accuracy = []
    precision = []
    recall = []
    f1 = []

    for train_index, test_index in tscv.split(X):

        # Split data

        X_train = X.iloc[train_index]
        X_test = X.iloc[test_index]

        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]

        # Train model

        model.fit(X_train, y_train)

        # Prediction

        prediction = model.predict(X_test)

        # Store metrics

        accuracy.append(
            accuracy_score(y_test, prediction)
        )

        precision.append(
            precision_score(
                y_test,
                prediction,
                zero_division=0
            )
        )

        recall.append(
            recall_score(
                y_test,
                prediction,
                zero_division=0
            )
        )

        f1.append(
            f1_score(
                y_test,
                prediction,
                zero_division=0
            )
        )

    results.append({

        "Model": model_name,

        "Accuracy Mean": np.mean(accuracy),
        "Accuracy Std": np.std(accuracy),

        "Precision Mean": np.mean(precision),
        "Precision Std": np.std(precision),

        "Recall Mean": np.mean(recall),
        "Recall Std": np.std(recall),

        "F1 Mean": np.mean(f1),
        "F1 Std": np.std(f1)

    })

## Naive Baseline
Always predict the market will go UP.

In [ ]:
baseline_accuracy = []
baseline_precision = []
baseline_recall = []
baseline_f1 = []

for train_index, test_index in tscv.split(X):

    y_test = y.iloc[test_index]

    # Always predict UP

    baseline_prediction = np.ones(len(y_test), dtype=int)

    baseline_accuracy.append(
        accuracy_score(y_test, baseline_prediction)
    )

    baseline_precision.append(
        precision_score(
            y_test,
            baseline_prediction,
            zero_division=0
        )
    )

    baseline_recall.append(
        recall_score(
            y_test,
            baseline_prediction,
            zero_division=0
        )
    )

    baseline_f1.append(
        f1_score(
            y_test,
            baseline_prediction,
            zero_division=0
        )
    )

results.append({

    "Model":"Naive Baseline",

    "Accuracy Mean":np.mean(baseline_accuracy),
    "Accuracy Std":np.std(baseline_accuracy),

    "Precision Mean":np.mean(baseline_precision),
    "Precision Std":np.std(baseline_precision),

    "Recall Mean":np.mean(baseline_recall),
    "Recall Std":np.std(baseline_recall),

    "F1 Mean":np.mean(baseline_f1),
    "F1 Std":np.std(baseline_f1)

})

## Comparison Table

In [ ]:
results_df = pd.DataFrame(results)

results_df

In [ ]:
ranking = results_df.sort_values(
    by="F1 Mean",
    ascending=False
)

ranking

In [84]:
ranking.to_csv(
    "model_ranking.csv",
    index=False
)

print("Evaluation completed successfully.")

Evaluation completed successfully.


# Model Evaluation Summary

The models were evaluated using a 5-fold TimeSeriesSplit to preserve
the chronological order of stock market data.

The Random Forest model achieved the highest F1 Score, indicating the
best balance between precision and recall.

Decision Tree performed reasonably well but showed signs of overfitting.

Logistic Regression served as a strong linear baseline.

The Naive Baseline (always predicting UP) performed worse than the
machine learning models, demonstrating that the trained models learned
useful patterns from the engineered features.